# Améliorations — reconstruire la trajectoire de chaque patient

Après le guide, notre meilleur modèle était à **7,43** (RMSE, validation par patient). Ce notebook explique comment nous sommes descendus à **3,37** (Kaggle : 3,339 public, 3,192 privé).

Les variables et le modèle sont dans `features.py` (dans ce même dossier). Ce notebook remplace les scripts `archives_py/experiment_day2_step8*.py` à `step10.py`.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")         # masque les avertissements de versions de librairies

# Le notebook est dans notebooks_finaux/ : on se place à la racine du repo pour lire data/,
# et on ajoute notebooks_finaux/ au chemin Python pour importer cv_splits.py et features.py.
if not os.path.exists("data"):
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "notebooks_finaux"))

import numpy as np
import pandas as pd
from sklearn.base import clone            # clone(modèle) = copie du modèle, non entraînée
import skore
from skore import evaluate                # evaluate = entraîne, teste et calcule les métriques

skore.configuration.show_progress = False  # pas de barres de progression dans le notebook

# Les rapports sont déjà sur Skore Hub : PUSH = False évite de les renvoyer en double.
PUSH = False

def push(key, report):
    """Envoie un rapport sur Skore Hub sous le nom `key`, seulement si PUSH vaut True."""
    if not PUSH:
        print(f"(PUSH=False) rapport {key!r} déjà sur le Hub, non renvoyé")
        return
    from skore import Project, login
    cfg = json.load(open(".skore", encoding="utf-8"))          # adresse du Hub, clé API, workspace
    os.environ["SKORE_HUB_URI"], os.environ["SKORE_HUB_API_KEY"] = cfg["hub_url"], cfg["api_key"]
    login(mode="hub")
    Project(name="bobathon-esilv", mode="hub", workspace=cfg["workspace"]).put(key, report)

In [2]:
def cv_rmse(report, data_source="test"):
    """RMSE de chacun des 5 folds d'un rapport de validation croisée (test ou train)."""
    return report.metrics.rmse(aggregate=None, data_source=data_source).iloc[0].to_numpy(dtype=float)

def show(name, report):
    """Affiche le RMSE moyen ± écart entre folds, et l'écart train/test (signe de surapprentissage)."""
    test, train = cv_rmse(report), cv_rmse(report, "train")
    print(f"{name:<22} test {test.mean():.3f} ± {test.std(ddof=1):.3f}   "
          f"train {train.mean():.3f}   écart {test.mean() - train.mean():+.3f}")
    return test

In [3]:
from cv_splits import load_visits, make_grouped_cv_splits

visits = load_visits()
y = visits["target"]
cv_splits = make_grouped_cv_splits(visits)   # exactement les mêmes 5 folds par patient qu'au Day 2

## 1. L'idée clé

Pour un patient, le vrai score OFF évolue **lentement et régulièrement** avec l'âge. Vérification : on ajuste une courbe du 2ᵉ degré de `target` en fonction de l'âge, patient par patient, et on mesure l'erreur restante.

In [4]:
def within_patient_rmse(deg):
    """Erreur restante quand on ajuste, patient par patient, une courbe de degré deg de target vs âge."""
    res = []
    for _, g in visits.groupby("patient_id"):            # g = toutes les visites d'un patient
        c = np.polyfit(g["age"], g["target"], deg)        # coefficients de la courbe
        res.append(g["target"] - np.polyval(c, g["age"]))  # écart entre la cible et la courbe
    return np.sqrt(np.mean(np.concatenate(res) ** 2))

for deg in [1, 2]:   # 1 = droite, 2 = parabole
    print(f"courbe de degré {deg} par patient : erreur {within_patient_rmse(deg):.2f}")

courbe de degré 1 par patient : erreur 1.61


courbe de degré 2 par patient : erreur 0.33


Une erreur de **0,33** : la cible est presque exactement une courbe lisse par patient. Les mesures `on` et `off`, elles, sont bruitées et biaisées :
- `off` est presque toujours **en dessous** du vrai score (le médicament restant fait baisser le score) ;
- `on` vaut environ 44 à 72 % du vrai score selon le délai depuis la prise.

Chaque patient du **test** a aussi ~8 visites : on peut combiner ses visites pour reconstruire sa trajectoire. **Règle anti-fuite :** on n'utilise jamais `target` dans une variable (il n'existe pas dans le test), et jamais `patient_id` comme variable.

In [5]:
# Écart entre off mesuré et vrai score : surtout négatif, donc off sous-estime le vrai score
(visits["off"] - visits["target"]).describe().round(2)

count    25677.00
mean        -5.95
std          8.03
min        -54.40
25%        -10.50
50%         -4.00
75%          0.00
max         14.50
dtype: float64

## 2. Les variables par patient (`features.py`)

`make_features` s'applique **de la même façon** au train et au test. Familles de variables :

| Famille | Exemples | Idée |
|---|---|---|
| Position dans le temps | `dur`, `t_first`, `visit_rank`, `n_visits` | où en est le patient dans sa maladie |
| Manques | `off_missing`, `on_missing` | un examen sauté est une information |
| `on` corrigé | `on_corr` | `on` divisé par l'effet attendu du médicament selon le délai |
| Statistiques du patient | `off_pmean`, `off_pmax`, `off_lin`, `off_quad`… | moyenne, max, droite et courbe ajustées sur ses visites |
| Visites voisines | `off_prev`, `off_next`… | mesure précédente et suivante |
| Trajectoire | `traj`, `traj_slope` | droite ajustée sur `off` + 5,9 et `on_corr`, pente tirée vers 2,8 points/an |
| Enveloppe supérieure | `off_env`, `comb_env`… | droite qui suit le **haut** des mesures `off` (puisque `off` sous-estime) |

In [6]:
from features import make_features, TwoStageRegressor

# target est retiré AVANT de construire les variables : aucune variable ne peut l'utiliser
X_feat = make_features(visits.drop(columns="target"))

# Variables d'enveloppe : retirées pour les modèles 08 et 09, ajoutées au modèle 10
ENV = ["off_env", "on_env", "comb_env", "off_minus_env", "traj_minus_env"]
print(X_feat.shape[1], "variables :", ", ".join(X_feat.columns))

47 variables : cohort, sexM, gene, age_at_diagnosis, age, ledd, time_since_intake_on, time_since_intake_off, on, off, dur, t_first, visit_rank, n_visits, off_missing, on_missing, on_corr, on_off_ratio, off_pmean, off_pmax, off_pcount, off_lin, off_quad, on_pmean, on_pmax, on_pcount, on_lin, on_quad, on_corr_pmean, on_corr_pmax, on_corr_pcount, on_corr_lin, on_corr_quad, off_prev, off_next, on_prev, on_next, on_corr_prev, on_corr_next, ledd_pmean, traj, traj_slope, off_env, on_env, comb_env, off_minus_env, traj_minus_env


## 3. Modèle 08 — HGBR sur les variables par patient

Même HGBR qu'au Day 2, un peu plus régularisé (`min_samples_leaf=40`, `l2_regularization=1.0`), sans les variables d'enveloppe (ajoutées plus tard).

In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Plus d'arbres (800) mais plus petits pas (0.03), et des feuilles d'au moins 40 visites
# + pénalité l2 : le modèle est plus précis sans trop mémoriser
hgbr = HistGradientBoostingRegressor(random_state=0, max_iter=800, learning_rate=0.03,
                                     min_samples_leaf=40, l2_regularization=1.0)
report_08 = evaluate(hgbr, X_feat.drop(columns=ENV), y, splitter=cv_splits)
show("08 variables patient", report_08)
push("08_patient_feats_gcv", report_08)

08 variables patient   test 3.740 ± 0.030   train 2.457   écart +1.282
(PUSH=False) rapport '08_patient_feats_gcv' déjà sur le Hub, non renvoyé


## 4. Modèle 09 — empilement en deux étages + lissage

`TwoStageRegressor` (dans `features.py`) :
1. **Étage 1** : le HGBR du modèle 08 prédit chaque visite. Ses prédictions sur le train sont faites **hors échantillon** (validation croisée interne par patient), sinon l'étage 2 apprendrait sur des prédictions trop parfaites.
2. **Étage 2** : un second HGBR reçoit en plus la trajectoire de ces prédictions pour le patient (moyenne, droite, courbe).
3. **Lissage** : les prédictions finales d'un patient sont remplacées par une courbe du 2ᵉ degré, puis ramenées entre 0 et 132.

`patient_id` est passé en colonne d'aide (pour regrouper les visites) mais retiré avant chaque modèle. ⏱ Quelques minutes.

In [8]:
X_in = X_feat.copy()
# patient_id est une colonne d'aide : TwoStageRegressor s'en sert pour regrouper les visites
# d'un patient, puis la retire avant chaque modèle (ce n'est jamais une variable)
X_in["patient_id"] = visits["patient_id"].to_numpy()

report_09 = evaluate(TwoStageRegressor(), X_in.drop(columns=ENV), y, splitter=cv_splits)   # sans enveloppe
show("09 empilement", report_09)
push("09_stack_gcv", report_09)

09 empilement          test 3.430 ± 0.021   train 1.748   écart +1.683
(PUSH=False) rapport '09_stack_gcv' déjà sur le Hub, non renvoyé


## 5. Modèle 10 — ajout de l'enveloppe supérieure

Puisque `off` est presque toujours **sous** le vrai score, la droite qui suit le haut des mesures `off` d'un patient est une meilleure estimation que leur moyenne. ⏱ Quelques minutes.

In [9]:
# Même modèle que 09, avec toutes les variables (enveloppe comprise)
report_10 = evaluate(TwoStageRegressor(), X_in, y, splitter=cv_splits)
folds_10 = show("10 + enveloppe", report_10)
print("10 meilleur que 09 sur", int((folds_10 < cv_rmse(report_09)).sum()), "folds sur 5")
push("10_env_stack_gcv", report_10)

10 + enveloppe         test 3.369 ± 0.049   train 1.701   écart +1.669
10 meilleur que 09 sur 4 folds sur 5
(PUSH=False) rapport '10_env_stack_gcv' déjà sur le Hub, non renvoyé


## 6. Soumission du modèle 10
Réentraînement sur tout le train, mêmes variables sur le test, fichier dans l'ordre de `sample_submission.csv`. ⏱ Une minute.

In [10]:
X_test = pd.read_csv("data/X_test.csv")
sample = pd.read_csv("data/sample_submission.csv")   # donne l'ordre des lignes attendu par Kaggle

# Même fonction que pour le train : les variables par patient sont calculées sur les visites du test
X_test_in = make_features(X_test)
X_test_in["patient_id"] = X_test["patient_id"].to_numpy()
X_test_in = X_test_in[X_in.columns]                  # mêmes colonnes, même ordre que le train

final = clone(TwoStageRegressor()).fit(X_in, y)      # entraîné sur toutes les visites du train
submission = pd.DataFrame({"Index": X_test["Index"], "target": final.predict(X_test_in)})
submission = submission.set_index("Index").reindex(sample["Index"]).reset_index()   # ordre de Kaggle

# Contrôles avant d'écrire le fichier : 11 013 lignes, pas de NaN, valeurs entre 0 et 132
assert len(submission) == 11013
assert submission["target"].notna().all()
assert submission["target"].between(0, 132).all()
submission.to_csv("submissions/10_env_stack.csv", index=False)
submission["target"].describe().round(2)

count    11013.00
mean        36.73
std         16.28
min          0.00
25%         25.00
50%         36.57
75%         48.52
max         93.95
Name: target, dtype: float64

## 7. Bilan

In [11]:
# Résumé : scores des notebooks 01 et 02 (repris) et des modèles calculés ci-dessus
bilan = pd.DataFrame({
    "modèle": ["01 dummy", "02 ridge", "07 tabular_pipeline (fin du guide)",
               "08 variables patient", "09 empilement + lissage", "10 + enveloppe"],
    "RMSE GroupKFold": [16.50, 10.44, 7.43,
                        cv_rmse(report_08).mean(), cv_rmse(report_09).mean(), cv_rmse(report_10).mean()],
})
bilan["RMSE GroupKFold"] = bilan["RMSE GroupKFold"].round(3)
bilan

,modèle,RMSE GroupKFold
0,01 dummy,16.500
1,02 ridge,10.440
2,07 tabular_pipeline (fin du guide),7.430
3,08 variables patient,3.740
4,09 empilement + lissage,3.430
5,10 + enveloppe,3.369


| | RMSE |
|---|---|
| Validation par patient (GroupKFold) | 3,369 |
| Découpage aléatoire | 3,074 (trop optimiste) |
| **Kaggle public** | **3,339** |
| **Kaggle privé** | **3,192** |

---
## Ce qu'il faut retenir

1. **La bonne validation.** La GroupKFold par patient prédit le score Kaggle à 0,03 près ; le découpage aléatoire est trop optimiste. Tous nos choix ont été faits sur la GroupKFold, jamais sur le classement public.
2. **Le gros gain vient des variables, pas du modèle.** Passer à des variables par patient divise l'erreur par deux (7,43 → 3,74) : on traite enfin la progression dans le temps, problème cité dans `docs/CONTEXT.md`.
3. **Comprendre comment les mesures sont biaisées paie.** `off` sous-estime le vrai score, d'où l'enveloppe supérieure ; `on` dépend du délai depuis la prise, d'où `on_corr`.
4. **Pas de fuite.** Aucune variable n'utilise `target` ; `patient_id` et `Index` ne sont jamais des variables (l'étape 6 a montré qu'ils font surapprendre) ; les prédictions de l'étage 1 sont hors échantillon.
5. **Surapprentissage sous contrôle.** Le score varie très peu d'un fold à l'autre (± 0,05) et le score privé (3,19) est même meilleur que le public.